# SAM2 forhåndsberegning
Colab kjører YOLO + SAM2 på alle bilder og lagrer overlay-bilder + polygon-data.
Etterpå gjennomgår du resultatet **lokalt** med `py -3.14 -m src.labeling.review_precomputed`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

REPO_URL  = 'https://github.com/davgei/trash-bin-detection.git'
BRANCH    = 'streetview-yolo-retry'
REPO_DIR  = '/content/trash-bin-detection'
DRIVE_ZIP = '/content/drive/MyDrive/data.zip'  # endre hvis zip ligger i en undermappe
YOLO_CONF = 0.20

# Klon eller oppdater repo
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}

In [ ]:
# Pakk ut data.zip
!unzip -q -o "$DRIVE_ZIP" -d "/content/trash-bin-detection"
!ls data/

In [ ]:
!pip install -q ultralytics
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
from ultralytics import YOLO, SAM

yolo = YOLO('models/trained/trash_bin_yolo11n_best.pt')
sam  = SAM('sam2.1_b.pt')   # lastes ned automatisk første gang
print('Modeller lastet.')

In [ ]:
import json
import cv2
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

ANNOTATED_DIR = Path('data/annotated')
SEG_DIR       = Path('data/annotated_seg')
PRECOMP_DIR   = Path('data/sam2_precomputed')
PRECOMP_DIR.mkdir(exist_ok=True)

def seg_label_path(split: str, stem: str) -> Path:
    return SEG_DIR / 'labels' / split / (stem + '.txt')

# Samle alle bilder på tvers av splits som ikke er prosessert ennå
to_process = []
for split in ['train', 'val', 'test']:
    img_dir = ANNOTATED_DIR / 'images' / split
    if not img_dir.exists():
        continue
    for img_path in sorted(img_dir.glob('*.jpg')) + sorted(img_dir.glob('*.png')):
        if (PRECOMP_DIR / img_path.name).exists():
            continue   # allerede forhåndsberegnet
        if seg_label_path(split, img_path.stem).exists():
            continue   # allerede har seg-etikett
        lbl = ANNOTATED_DIR / 'labels' / split / (img_path.stem + '.txt')
        to_process.append((img_path, lbl, split))

print(f'Bilder i annotated/: {sum(1 for s in ["train","val","test"] for _ in (ANNOTATED_DIR / "images" / s).glob("*.jpg"))}')
print(f'Skal prosesseres:    {len(to_process)}')

In [ ]:
def read_yolo_boxes(label_path: Path, img_w: int, img_h: int) -> list:
    """Les eksisterende YOLO-bokselabels og konverter til pikselkoordinater."""
    if not label_path.exists() or label_path.stat().st_size == 0:
        return []
    boxes = []
    for line in label_path.read_text(encoding='utf-8').strip().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        x1 = (cx - bw / 2) * img_w
        y1 = (cy - bh / 2) * img_h
        x2 = (cx + bw / 2) * img_w
        y2 = (cy + bh / 2) * img_h
        boxes.append([x1, y1, x2, y2])
    return boxes


def mask_to_polygon(mask, img_w, img_h):
    contours, _ = cv2.findContours(
        (mask > 0.5).astype(np.uint8) * 255,
        cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) < 100:
        return None
    approx = cv2.approxPolyDP(c, 0.005 * cv2.arcLength(c, True), True)
    pts = approx.reshape(-1, 2).astype(float)
    pts[:, 0] /= img_w
    pts[:, 1] /= img_h
    return pts.flatten().tolist()


for img_path, lbl_path, split in tqdm(to_process):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    overlay  = img.copy()
    polygons = []

    # Bruk eksisterende YOLO-bokselabels som SAM2-prompt
    boxes = read_yolo_boxes(lbl_path, w, h)

    if boxes:
        seg   = sam(str(img_path), bboxes=boxes, verbose=False)[0]
        masks = seg.masks.data.cpu().numpy() if seg.masks is not None else []
        for mask in masks:
            poly = mask_to_polygon(mask, w, h)
            if poly:
                polygons.append(poly)
            colored = np.zeros_like(img)
            colored[mask > 0.5] = (0, 200, 60)
            cv2.addWeighted(overlay, 1.0, colored, 0.45, 0, dst=overlay)
        for box in boxes:
            cv2.rectangle(overlay,
                (int(box[0]), int(box[1])), (int(box[2]), int(box[3])),
                (0, 230, 0), 2)
    else:
        cv2.rectangle(overlay, (0, 0), (w-1, h-1), (0, 0, 220), 8)
        cv2.putText(overlay, 'INGEN BOKSER I LABEL', (10, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 0, 220), 4)

    cv2.imwrite(str(PRECOMP_DIR / img_path.name), overlay)
    (PRECOMP_DIR / (img_path.stem + '.json')).write_text(
        json.dumps({'polygons': polygons, 'n_detections': len(boxes), 'split': split}),
        encoding='utf-8')

print(f'Ferdig! {len(to_process)} bilder lagret i {PRECOMP_DIR}')

In [ ]:
# Lagre oppdatert data.zip (inkl. sam2_precomputed/) til Drive
import shutil
shutil.make_archive('/content/data_updated', 'zip', '/content/trash-bin-detection', 'data')
shutil.move('/content/data_updated.zip', '/content/drive/MyDrive/data.zip')
print('Lastet opp til Drive som data.zip')
print()
print('Neste steg lokalt:')
print('  py -3.14 -m src.labeling.review_precomputed')
print('  py -3.14 -m src.labeling.export_labels')
print('  py -3.14 -m src.prepare_dataset')